# Creating a Labeled Dataset for the Artmatcher

This notebook explains how to build a labeled dataset that the artmatcher's Random Forest classifier can be trained on.

The process is **iterative**: you start with a simple classifier, label examples, train a better model, then batch-classify with that model, then label more.

```mermaid
flowchart LR
  A[Embeddings] -->|first pass: no model yet| C[serve labeling UI]
  A --> B[batch-classify\nrequires trained model]
  B --> C
  C -->|label examples| D[dataset .json]
  D --> E[train-classifier]
  E -->|model ready| B
```

**Important:** `batch-classify` requires `--ml-model` (a trained model). On the very first pass, skip it — go directly to `serve --classifier simple`.

**What you need before starting:**
- A `.pkl` embeddings file for your reference images (WikiArt or similar)
- The Glashelder images directory (`ASSETS_DIR`)
- Optionally: pre-computed Glashelder embeddings (speeds things up a lot)

## Setup

In [ ]:
import subprocess
import time
from pathlib import Path

from IPython.display import HTML, display

### Configuration

Set these paths before running anything else.

In [ ]:
# Required: path to your reference image embeddings (.pkl)
WIKIART_EMBEDDINGS = Path(
    "/mnt/share/shares/glashelder/embeddings/wikiart-dino.pkl"
)  # e.g. /data/wikiart_clip_embeddings.pkl

# Optional: pre-computed Glashelder embeddings (.pkl) — much faster than recomputing
GLASHELDER_EMBEDDINGS = Path(
    "/mnt/share/shares/glashelder/embeddings/glashelder-dino.pkl"
)

# everything below in this cell, you don't need to edit
# but you can if you want!

# Where to save batch-classify output
BATCH_RESULTS = Path("./batch_results.json")

# Where to save the labeled dataset exported from the UI
LABELED_DATASET = Path("./labeling_dataset.json")

# Where to save the trained model
TRAINED_MODEL = Path("./trained-matcher-model.pkl")

# Secret for the labeling UI — set to any string you like
GLASHELDER_SECRET = "mysecret"

# Port for the labeling UI
PORT = 5000

---

## Step 1: Run `batch-classify` (requires a trained model)

`batch-classify` processes **all** Glashelder images through the matcher and saves results to JSON.
The labeling UI can load this pre-computed file instead of computing on the fly — much faster.

> **First pass?** Skip this step. You need a trained model first.
> Jump to [Step 2](#Step-2:-Launch-the-Labeling-UI) and use `serve --classifier simple` instead.

**What it does:**
1. Loads (or generates) CLIP embeddings for all Glashelder images
2. For each image, finds the top-K most similar reference images by cosine similarity
3. Extracts 23 ML features (distribution stats + ORB geometry + perceptual hashes)
4. Runs the ML classifier → confidence score + `is_match`
5. Saves everything to JSON

**Output format** (one entry per Glashelder image):
```json
{
  "results": [
    {
      "glashelder_path": "/data/glashelder/img_001.jpg",
      "glashelder_index": 0,
      "wikiart_path": "/data/wikiart/monet_water_lilies.jpg",
      "wikiart_index": 42,
      "similarities": [0.91, 0.42, 0.38, 0.35, 0.33, 0.31, 0.29, 0.27, 0.25, 0.23],
      "is_match": true,
      "confidence": 0.87,
      "threshold": 0.5,
      "features": {
        "sim_0": 0.91,
        "gap_01": 0.49,
        "orb_num_inliers": 12,
        "phash_distance": 8
      }
    }
  ]
}
```

In [4]:
# Build the batch-classify command.
# Requires a trained model — skip this on first pass, go to Step 2 instead.

if not TRAINED_MODEL.exists():
    print(f"No trained model at {TRAINED_MODEL}")
    print("Skip to Step 2 and label examples first, then train in Step 4.")
else:
    cmd = [
        "uv",
        "run",
        "glashelder",
        "artmatcher",
        "batch-classify",
        "--wikiart-embeddings",
        str(WIKIART_EMBEDDINGS),
        "--ml-model",
        str(TRAINED_MODEL),
        "--output",
        str(BATCH_RESULTS),
        "--workers",
        "8",
        "--top-k",
        "10",
        "--threshold",
        "0.5",
    ]

    if GLASHELDER_EMBEDDINGS is not None:
        cmd += ["--glashelder-embeddings", str(GLASHELDER_EMBEDDINGS)]

    print("Command:")
    print(" ".join(cmd))

No trained model at trained-matcher-model.pkl
Skip to Step 2 and label examples first, then train in Step 4.


In [ ]:
# Uncomment to run (takes a while for large datasets)
# result = subprocess.run(cmd, capture_output=True, text=True)
# print(result.stdout)
# if result.returncode != 0:
#     print("STDERR:", result.stderr)

Or from the terminal:

```bash
uv run glashelder artmatcher batch-classify \
  --wikiart-embeddings /path/to/wikiart.pkl \
  --ml-model trained-matcher-model.pkl \
  --output batch_results.json \
  --workers 8
```

---

## Step 2: Launch the Labeling UI

The labeling UI is a Flask web app where you review candidate matches and label them as:

| Button | Meaning | Stored as |
|--------|---------|----------|
| ✓ Correct | Classifier was right | `positives` (match) or `negatives` (no match) |
| ✗ Incorrect | Classifier was wrong | `false_positives` or `false_negatives` |

The UI also lets you:
- Browse matches and non-matches separately
- Adjust the confidence threshold in real time
- Download the labeled dataset as JSON

**Authentication:** The UI requires a secret. Access it at `http://localhost:<PORT>/?secret=<GLASHELDER_SECRET>` once to set a session cookie.

In [23]:
import os

# Build the serve command
serve_cmd = [
    "uv",
    "run",
    "glashelder",
    "artmatcher",
    "serve",
    "--port",
    str(PORT),
]

if BATCH_RESULTS.exists():
    serve_cmd += ["--results-json", str(BATCH_RESULTS)]
    print(f"Loading pre-computed results from {BATCH_RESULTS}")
else:
    # Fall back to on-the-fly computation
    serve_cmd += [
        "--wikiart-embeddings",
        str(WIKIART_EMBEDDINGS),
    ]
    if GLASHELDER_EMBEDDINGS is not None:
        serve_cmd += ["--glashelder-embeddings", str(GLASHELDER_EMBEDDINGS)]
    if TRAINED_MODEL.exists():
        serve_cmd += ["--classifier", "ml", "--ml-model", str(TRAINED_MODEL)]
    print("No pre-computed results found — computing on the fly (slower).")
    print("Run batch-classify first for a much faster experience.")

print("Command:")
print(" ".join(serve_cmd))

No pre-computed results found — computing on the fly (slower).
Run batch-classify first for a much faster experience.
Command:
uv run glashelder artmatcher serve --port 5000 --wikiart-embeddings /mnt/share/shares/glashelder/embeddings/wikiart-dino.pkl --glashelder-embeddings /mnt/share/shares/glashelder/embeddings/glashelder-dino.pkl


In [24]:
# Start the Flask server in the background
env = os.environ.copy()
env["GLASHELDER_SECRET"] = GLASHELDER_SECRET

server_process = subprocess.Popen(
    serve_cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for Flask to start
time.sleep(3)

if server_process.poll() is not None:
    out, _ = server_process.communicate()
    print("Server failed to start:")
    print(out.decode())
else:
    print(f"Server running (PID {server_process.pid})")

Server running (PID 23885)


In [ ]:
# Open the labeling UI
# The ?secret= parameter authenticates you and sets a session cookie — only needed once.

url = f"http://localhost:{PORT}/?secret={GLASHELDER_SECRET}"

display(
    HTML(
        f'<p><strong>Labeling UI:</strong> <a href="{url}" target="_blank" style="font-size:1.1em"> Open in new tab ↗ </a> </p>'
    )
)

In [26]:
# Stop the server when done labeling
server_process.terminate()
print("Server stopped.")

Server stopped.


---

## Step 3: The Labeled Dataset

After labeling, click **"Download Dataset"** in the UI. Save the file to `LABELED_DATASET` path configured above.

The dataset JSON has four categories:

```json
{
  "positives": [...],       // classifier predicted match → you confirmed correct
  "negatives": [...],       // classifier predicted no match → you confirmed correct
  "false_positives": [...], // classifier predicted match → you marked incorrect
  "false_negatives": [...]  // classifier predicted no match → you marked incorrect
}
```

Each entry looks like this:

```json
{
  "glashelder_path": "/data/glashelder/img_001.jpg",
  "wikiart_path": "/data/wikiart/monet_water_lilies.jpg",
  "similarities": [0.91, 0.42, 0.38, 0.35, 0.33, 0.31, 0.29, 0.27, 0.25, 0.23],
  "glashelder_index": 0,
  "wikiart_index": 42,
  "diff": 0.87,
  "threshold": 0.5
}
```

The `similarities` list is what the classifier uses — it's the top-K cosine similarity scores between the Glashelder image and reference images, in descending order.

In [ ]:
# Inspect the dataset
import json

if LABELED_DATASET.exists():
    with open(LABELED_DATASET) as f:
        dataset = json.load(f)

    for category, entries in dataset.items():
        print(f"{category:20s}: {len(entries):4d} examples")

    total = sum(len(v) for v in dataset.values())
    n_pos = len(dataset["positives"]) + len(dataset["false_negatives"])
    n_neg = len(dataset["negatives"]) + len(dataset["false_positives"])
    print(f"\nTotal: {total}  |  True matches: {n_pos}  |  Non-matches: {n_neg}")
else:
    print(f"No dataset found at {LABELED_DATASET}")
    print("Download it from the labeling UI first.")

---

## Step 4: Train the Classifier

Once you have a labeled dataset, train a Random Forest classifier on it.
The classifier learns to distinguish true matches from false positives using the 23 features.

**`train-classifier` options:**
- `--n-estimators` — number of trees (more = better but slower; 100 is a good default)
- `--cv-folds` — cross-validation folds for accuracy estimate
- `--workers` — parallel CPU workers for ORB/hash feature extraction
- `--no-image-features` — skip ORB/hash extraction (faster, less accurate)

In [ ]:
from artmatcher.ml_classifier import MLClassifier

if LABELED_DATASET.exists():
    classifier = MLClassifier(threshold=0.5)

    metrics = classifier.train(
        dataset_path=str(LABELED_DATASET),
        n_estimators=100,
        cv_folds=5,
        extract_image_features=True,
        workers=4,
    )

    print(
        f"Cross-validation accuracy: {metrics.cv_accuracy:.3f} ± {metrics.cv_std:.3f}"
    )
    print(
        f"Samples: {metrics.n_samples} ({metrics.n_positives} pos, {metrics.n_negatives} neg)"
    )

    classifier.save(str(TRAINED_MODEL))
    print(f"\nModel saved to {TRAINED_MODEL}")
    print("Re-run Step 1 (batch-classify) with this model to get better results.")
else:
    print(f"No dataset at {LABELED_DATASET} — label examples in Step 2 first.")

Or from the CLI:

```bash
uv run glashelder artmatcher train-classifier \
  labeling_dataset.json \
  trained-matcher-model.pkl \
  --workers 8
```

---

## The Iterative Loop

Dataset quality improves with each iteration:

1. **First pass** — skip `batch-classify`. Run `serve --classifier simple` directly. Label ~50-100 examples.
2. **Train** — `train-classifier` on labeled data → get first model
3. **Batch classify** — run `batch-classify --ml-model` with the new model → pre-computed results JSON
4. **Label more** — load results into `serve --results-json`. The ML model surfaces harder edge cases; label those.
5. **Repeat** from step 2 until accuracy plateaus

**Labeling tips:**
- Focus on **false positives** (classifier thinks it's a match but isn't) — hardest cases, most valuable
- Adjust the threshold slider in the UI to expose more borderline cases
- Aim for a balanced dataset: roughly equal positives and negatives
- 200+ examples is usually enough for a good first model; 500+ for a robust one